In [50]:
import numpy as np
import pandas as pd
import mlflow
import onnx
import os
import onnxruntime as rt
from onnxmltools import convert_xgboost
from onnxmltools.utils import save_model
from onnxmltools.convert.common.data_types import FloatTensorType
import sys
sys.path.append('..')
from src.train import get_mlflow_client
import time
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

In [8]:
mlflow_client = get_mlflow_client("http://127.0.0.1:5000", "xgboost_training")

In [9]:
model = mlflow.xgboost.load_model("models:/xgboost_tsd_model@champion")

In [10]:
booster = model.get_booster()
booster.feature_names = None

In [11]:
onnx_model = convert_xgboost(
    model, 
    initial_types=[('input', FloatTensorType([None, model.n_features_in_]))]
)

In [ ]:
models_dir = os.path.abspath("models")
os.makedirs(models_dir, exist_ok=True)
save_model(onnx_model, os.path.join(models_dir, "xgboost_tsd_model.onnx"))
print(f"Saved to: {os.path.join(models_dir, 'xgboost_tsd_model.onnx')}")

Saved to: C:\Users\ranas\OneDrive\Desktop\TSD\models\xgboost_tsd_model.onnx


In [15]:

runtime = rt.InferenceSession(os.path.join(models_dir, "xgboost_tsd_model.onnx"))
input_name = runtime.get_inputs()[0].name
print(f"Input name: {input_name}")

Input name: input


In [37]:
test_data = pd.read_csv('data/test_x.csv')
test_data = test_data.astype(np.float32)

In [ ]:
onnx_inputs = {input_name: test_data.values}
for _ in range(10):
    onnx_outputs = runtime.run(None, onnx_inputs)
start = time.perf_counter()
for _ in range(1000):
    onnx_outputs = runtime.run(None, onnx_inputs)
end = time.perf_counter()
elapsed_time_xgboost_onnx = end - start
print(f"ONNX Runtime inference time: {elapsed_time_xgboost_onnx:.4f} seconds")

ONNX Runtime inference time: 59.6786 seconds


In [ ]:
for _ in range(10):
    outputs = model.predict(test_data)
start = time.perf_counter()
for _ in range(1000):
    outputs = model.predict(test_data)
end = time.perf_counter()
elapsed_time_xgboost = end - start
print(f"XGBoost inference time: {elapsed_time_xgboost:.4f} seconds")

XGBoost inference time: 29.2390 seconds


In [45]:
speedup = elapsed_time_xgboost / elapsed_time_xgboost_onnx
print(f"ONNX Runtime is {speedup:.1f}x faster than native XGBoost")

ONNX Runtime is 0.5x faster than native XGBoost


In [53]:
model_proto = onnx.load(str(Path(models_dir) / "xgboost_tsd_model.onnx"))

# Add missing ai.onnx opset
opset = model_proto.opset_import.add()
opset.domain = "ai.onnx"
opset.version = 17

onnx.save(model_proto, str(Path(models_dir) / "xgboost_tsd_model_fixed.onnx"))
print("Fixed model saved")

Fixed model saved


In [54]:
quantize_dynamic(
    model_input=str(Path(models_dir) / "xgboost_tsd_model_fixed.onnx"),
    model_output=str(Path(models_dir) / "xgboost_tsd_model_quantized.onnx"),
    weight_type=QuantType.QInt8
)

In [56]:
size_in_bytes_onnx = Path(models_dir) / "xgboost_tsd_model_fixed.onnx"
print(f"ONNX model size: {size_in_bytes_onnx.stat().st_size / (1024 * 1024):.2f} MB")

ONNX model size: 0.23 MB


In [57]:
size_in_bytes_onnx = Path(models_dir) / "xgboost_tsd_model_quantized.onnx"
print(f"ONNX model size: {size_in_bytes_onnx.stat().st_size / (1024 * 1024):.2f} MB")

ONNX model size: 0.23 MB
